# 10 – Interpretation: rules learned by the decision tree

The random forest is accurate but hard to read. The shallow decision tree with physics features scored close to it in cross-validation (PR-AUC 0.886 vs 0.909, notebook 07) and can be printed as if-then rules. Comparing its split points with the failure thresholds documented for the dataset shows whether the models learned the real failure mechanisms.

The tree uses the settings chosen in every outer fold of notebook 07 (depth 5, at least 10 jobs per leaf, no class weighting) and is fitted on the training set only. It is not evaluated on the test set.

In [1]:
import pandas as pd
from sklearn.tree import export_text

from config import RESULTS_DIR
from data import load_train_test
from features import build_features
from models import make_pipeline

train, _ = load_train_test()
X_train, y_train = build_features(train)

tree = make_pipeline("decision_tree", "physics", max_depth=5, min_samples_leaf=10, class_weight=None).fit(X_train, y_train)
names = list(tree.named_steps["preprocess"].get_feature_names_out())

## 1. Rules

`weights: [a, b]` gives the number of training jobs without (a) and with (b) a failure in that leaf; `class` is the majority outcome.

In [2]:
rules = export_text(tree.named_steps["model"], feature_names=names, decimals=2, show_weights=True)
(RESULTS_DIR / "10_tree_rules.txt").write_text(rules)
print(rules)

|--- strain <= 10998.50
|   |--- power <= 9000.17
|   |   |--- power <= 3496.14
|   |   |   |--- weights: [0.00, 24.00] class: 1
|   |   |--- power >  3496.14
|   |   |   |--- temp_diff <= 8.65
|   |   |   |   |--- speed <= 1381.00
|   |   |   |   |   |--- weights: [9.00, 72.00] class: 1
|   |   |   |   |--- speed >  1381.00
|   |   |   |   |   |--- weights: [470.00, 3.00] class: 0
|   |   |   |--- temp_diff >  8.65
|   |   |   |   |--- tool_wear <= 202.50
|   |   |   |   |   |--- weights: [5907.00, 2.00] class: 0
|   |   |   |   |--- tool_wear >  202.50
|   |   |   |   |   |--- weights: [345.00, 25.00] class: 0
|   |--- power >  9000.17
|   |   |--- weights: [0.00, 35.00] class: 1
|--- strain >  10998.50
|   |--- type_L <= 0.50
|   |   |--- strain <= 11705.40
|   |   |   |--- weights: [13.00, 2.00] class: 0
|   |   |--- strain >  11705.40
|   |   |   |--- weights: [6.00, 6.00] class: 0
|   |--- type_L >  0.50
|   |   |--- weights: [0.00, 62.00] class: 1



## 2. Learned split points vs documented failure rules

In [3]:
fitted = tree.named_steps["model"].tree_
reference = {
    "strain": "11,000 / 12,000 / 13,000 min·Nm for L / M / H (OSF)",
    "power": "3,500 and 9,000 W (PWF)",
    "temp_diff": "8.6 K, together with speed (HDF)",
    "speed": "1,380 rpm, together with temperature difference (HDF)",
    "tool_wear": "documented 200–240 min, observed 198–253 min (TWF, random within this window)",
    "type_L": "type L has the lowest strain limit (OSF)",
}
splits = pd.DataFrame({
    "feature": [names[i] for i in fitted.feature if i >= 0],
    "learned threshold": [round(t, 2) for i, t in zip(fitted.feature, fitted.threshold) if i >= 0],
})
splits["documented rule"] = splits["feature"].map(reference)
splits.to_csv(RESULTS_DIR / "10_tree_splits.csv", index=False)
pd.set_option("display.max_colwidth", None)
splits

,feature,learned threshold,documented rule
0,strain,10998.50,"11,000 / 12,000 / 13,000 min·Nm for L / M / H (OSF)"
1,power,9000.17,"3,500 and 9,000 W (PWF)"
2,power,3496.14,"3,500 and 9,000 W (PWF)"
3,temp_diff,8.65,"8.6 K, together with speed (HDF)"
4,speed,1381.00,"1,380 rpm, together with temperature difference (HDF)"
5,tool_wear,202.50,"documented 200–240 min, observed 198–253 min (TWF, random within this window)"
6,type_L,0.50,type L has the lowest strain limit (OSF)
7,strain,11705.40,"11,000 / 12,000 / 13,000 min·Nm for L / M / H (OSF)"
